# prime-rl S4: LFM2.5-VL load + forward (RTX 6000 Pro offline)

Goal: confirm `LiquidAI/LFM2.5-VL-450M` loads via `AutoModelForImageTextToText`
from the prep-bundled snapshot, runs a forward pass on a dummy image+text
input, and emits a sane chat template (the latter matters for SatelliteAgent's
ReAct trace formatting later).

Success criteria:
- Model + processor load without error
- `model.generate(...)` produces non-empty output for a colored 224x224 image
- Chat template applies cleanly to a sample messages list

## 1. Install (same recipe as S3 minus reverse_text patch)

In [ ]:
import os, subprocess, glob, re

PREP = "/kaggle/input/notebooks/titanic12/prime-rl-offline-prep"
BASE = f"{PREP}/output" if os.path.exists(f"{PREP}/output") else PREP
WHEELS = f"{BASE}/wheels"
MODEL_DIR = f"{BASE}/models/LFM2.5-VL-450M"

# Keep Kaggle's preinstalled GPU stack AND scientific stack (numpy/scipy/etc).
# Our wheels have numpy 2.2.6 while Kaggle ships numpy 2.3+, and overwriting
# numpy breaks scipy (`cannot import name '_center' from numpy._core.umath`),
# which transformers.generation pulls in via sklearn.
SKIP_PREFIXES = (
    "torch-", "torchvision-", "torchaudio-", "nvidia_",
    "numpy-", "scipy-", "scikit_learn-", "pandas-",
)

def parse_version(name):
    m = re.match(r"([a-zA-Z0-9_]+?)-(\d+(?:\.\d+)*)", os.path.basename(name))
    if not m:
        return None, None
    return m.group(1).lower(), tuple(int(x) for x in m.group(2).split("."))

all_wheels = sorted(glob.glob(f"{WHEELS}/*.whl"))
remaining = [w for w in all_wheels if not os.path.basename(w).startswith(SKIP_PREFIXES)]
keep = {}
for w in remaining:
    pkg, ver = parse_version(w)
    if pkg is None:
        keep[os.path.basename(w)] = (None, w); continue
    if pkg in keep and keep[pkg][0] is not None:
        if ver > keep[pkg][0]:
            keep[pkg] = (ver, w)
    else:
        keep[pkg] = (ver, w)
filtered = [v[1] for v in keep.values()]
print(f"installing {len(filtered)} wheels")
subprocess.run(
    "python3.12 -m pip install --no-index --no-build-isolation --pre --no-deps "
    + " ".join(filtered),
    shell=True, check=True,
)

## 2. Sanity check

In [ ]:
import os, subprocess, torch, transformers
PREP = "/kaggle/input/notebooks/titanic12/prime-rl-offline-prep"
BASE = f"{PREP}/output" if os.path.exists(f"{PREP}/output") else PREP
MODEL_DIR = f"{BASE}/models/LFM2.5-VL-450M"

subprocess.run("nvidia-smi --query-gpu=name,memory.total --format=csv", shell=True)
print()
print(f"torch={torch.__version__}, cuda={torch.cuda.is_available()}")
print(f"transformers={transformers.__version__}")
print()
print("--- LFM2.5-VL-450M files ---")
subprocess.run(f"ls -la {MODEL_DIR}", shell=True)

## 3. Load model + processor

LFM2.5-VL is registered in transformers >=5.1 under the LFM2-VL architecture
and is loaded via `AutoModelForImageTextToText` + `AutoProcessor`.

In [ ]:
import os, torch
os.environ.setdefault("HF_HUB_OFFLINE", "1")
os.environ.setdefault("TRANSFORMERS_OFFLINE", "1")

PREP = "/kaggle/input/notebooks/titanic12/prime-rl-offline-prep"
BASE = f"{PREP}/output" if os.path.exists(f"{PREP}/output") else PREP
MODEL_DIR = f"{BASE}/models/LFM2.5-VL-450M"

from transformers import AutoModelForImageTextToText, AutoProcessor

processor = AutoProcessor.from_pretrained(MODEL_DIR, trust_remote_code=False)
print(f"processor: {type(processor).__name__}")

model = AutoModelForImageTextToText.from_pretrained(
    MODEL_DIR,
    dtype=torch.bfloat16,
    device_map="cuda",
    trust_remote_code=False,
)
n_params = sum(p.numel() for p in model.parameters()) / 1e6
print(f"model: {type(model).__name__}, {n_params:.1f}M params")
print(f"VRAM after load: {torch.cuda.memory_allocated() / 1e9:.2f} GB")

## 4. Generate from a dummy image

In [ ]:
from PIL import Image
import torch, time

# 224x224 solid blue image -- something the VLM should see clearly.
img = Image.new("RGB", (224, 224), color=(40, 80, 200))

messages = [
    {
        "role": "user",
        "content": [
            {"type": "image", "image": img},
            {"type": "text", "text": "What dominant color do you see?"},
        ],
    }
]

inputs = processor.apply_chat_template(
    messages,
    add_generation_prompt=True,
    tokenize=True,
    return_tensors="pt",
    return_dict=True,
).to(model.device, dtype=torch.bfloat16)

t0 = time.time()
out = model.generate(**inputs, max_new_tokens=64, do_sample=False)
elapsed = time.time() - t0
txt = processor.batch_decode(out[:, inputs["input_ids"].shape[-1]:], skip_special_tokens=True)[0]
print(f"generated in {elapsed:.1f}s\n--- output ---\n{txt}\n---")

## 5. Chat template inspection (for SatelliteAgent ReAct compat)

In [ ]:
tpl = getattr(processor, "chat_template", None) or getattr(processor.tokenizer, "chat_template", None)
print("--- chat_template (first 600 chars) ---")
print((tpl or "<no chat_template>")[:600])
print()

# Test with a 3-turn React-style message
react_messages = [
    {"role": "system", "content": "You are a satellite agent."},
    {"role": "user", "content": [
        {"type": "image", "image": img},
        {"type": "text", "text": "Classify the change."},
    ]},
    {"role": "assistant", "content": "Thought: image looks uniform. Action: drop()"},
]
rendered = processor.apply_chat_template(react_messages, tokenize=False, add_generation_prompt=False)
print("--- rendered React-style (truncated 800 chars) ---")
print(rendered[:800])
print()
print("S4 PASS")